# support-rag-agent: vLLM smoke test (Task 10)

Checks that `OpenAIChatClient` + the tool-calling loop work against a real open model
(`Qwen/Qwen3-8B-AWQ` served by vLLM) before the API and evaluation tasks.

**Kaggle setup (before running)**
1. Upload the `support-rag-agent` folder (without `.venv`, `__pycache__`, `.ruff_cache`) as a Kaggle
   Dataset, then attach it to this notebook (Add Input). Any dataset name works: the project is
   found by searching `/kaggle/input` for its `pyproject.toml` (exactly one must exist).
2. Settings > Accelerator: a **T4** GPU (T4 x2 is fine: vLLM is pinned to GPU 0).
3. Settings > Internet: **On** (pip install and the model download from Hugging Face).
4. Run the cells top to bottom. Install + model download + server start can take 10-15 min.
5. Copy the output of the last cell into `docs/dev_log.md` (section "vLLM smoke test (Task 10)").

No secrets are needed: the model is public and the local server ignores the API key.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

# bm25s imports jax when it is installed (Kaggle has it) and JAX would grab the GPU memory vLLM
# needs. Force CPU and no preallocation BEFORE anything imports support_agent or bm25s.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")

# Kaggle nests datasets (e.g. /kaggle/input/datasets/<user>/<slug>/), so search for the project.
matches = sorted(INPUT_DIR.rglob("pyproject.toml"))
assert len(matches) == 1, f"expected exactly one pyproject.toml under {INPUT_DIR}, got {matches}"
PROJECT_DIR = matches[0].parent
project = WORK_DIR / "support-rag-agent"
shutil.copytree(PROJECT_DIR, project, dirs_exist_ok=True)
sys.path.insert(0, str(project / "src"))  # no `pip install -e .`: it would replace the GPU torch
print("project copied to", project)

In [ ]:
# Not `pip install -e .`: pyproject pins CPU torch, which would break vLLM's GPU torch.
# Only the light deps the smoke path imports (openai, pydantic, bm25s, numpy) plus vLLM.
VLLM_VERSION = "0.32.0"  # exact pin; taken from the vLLM docs version list
!pip install -q "vllm=={VLLM_VERSION}" openai pydantic bm25s numpy

In [ ]:
import subprocess
import time
import urllib.request

MODEL = "Qwen/Qwen3-8B-AWQ"
PORT = 8000
READY_TIMEOUT_S = 15 * 60  # model download + load on a T4
POLL_INTERVAL_S = 5
LOG_TAIL_CHARS = 4000
LOG_PATH = WORK_DIR / "vllm.log"

log_file = open(LOG_PATH, "w")
server = subprocess.Popen(
    [
        "vllm",
        "serve",
        MODEL,
        "--port",
        str(PORT),
        "--dtype",
        "half",  # T4 has no bf16
        "--enable-auto-tool-choice",
        "--tool-call-parser",
        "hermes",
        "--max-model-len",
        "8192",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    # One GPU is enough for 8B AWQ; tensor parallelism across a T4 x2 is not needed.
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
)


def log_tail() -> str:
    return LOG_PATH.read_text(errors="replace")[-LOG_TAIL_CHARS:]


def wait_until_ready() -> None:
    deadline = time.monotonic() + READY_TIMEOUT_S
    while time.monotonic() < deadline:
        if server.poll() is not None:
            raise RuntimeError(f"vLLM exited with code {server.returncode}:\n{log_tail()}")
        try:
            with urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=5):
                return
        except OSError:
            time.sleep(POLL_INTERVAL_S)
    server.terminate()
    raise TimeoutError(f"vLLM not ready after {READY_TIMEOUT_S}s:\n{log_tail()}")


def ensure_server_alive() -> None:
    if server.poll() is not None:
        raise RuntimeError(f"vLLM died with code {server.returncode}:\n{log_tail()}")


wait_until_ready()
print("vLLM is ready")

In [ ]:
from datetime import timedelta

from support_agent.agent import AgentConfig, run_agent
from support_agent.corpus import chunk_passages, load_corpus
from support_agent.llm import OpenAIChatClient
from support_agent.orders import REFERENCE_NOW, connect, from_iso, generate, get_order
from support_agent.policy import COMPLAINT_WINDOW_DAYS
from support_agent.retrieval import build_retriever
from support_agent.tools import ToolContext

CANARY = "CANARY-smoke-7f3a"
NONCE = "smoke-n0nce42"

chunks = chunk_passages(load_corpus(project / "corpus"))
retrieve = build_retriever("bm25", chunks, None, None)  # no embedding models in the smoke test

db_path = WORK_DIR / "orders_smoke.db"
db_path.unlink(missing_ok=True)
conn = connect(db_path)
generate(conn)

COMPLAINT_ORDER_SQL = "SELECT id FROM orders WHERE status = 'entregue' ORDER BY delivered_at DESC"


def find_refundable_order() -> dict:
    """First delivered order, still inside its store type's complaint window, with no refund."""
    for (order_id,) in conn.execute(COMPLAINT_ORDER_SQL).fetchall():
        order = get_order(conn, order_id)
        age = REFERENCE_NOW - from_iso(order["delivered_at"])
        window = timedelta(days=COMPLAINT_WINDOW_DAYS[order["store_type"]])
        if order["refund"] is None and age <= window:
            return order
    raise RuntimeError("no refundable order in the generated data")


refund_order = find_refundable_order()
customer_id = refund_order["customer_id"]
refund_item = refund_order["items"][0]["nome"]
status_order_id = conn.execute(
    "SELECT id FROM orders WHERE customer_id = ? AND id != ? ORDER BY id LIMIT 1",
    (customer_id, refund_order["id"]),
).fetchone()["id"]

QUESTIONS = [
    {
        "q": "Posso desistir de uma compra de mercado feita pelo aplicativo? Em quantos dias?",
        "tool": False,
    },
    {"q": "O que acontece se meu pedido atrasar muito? Tenho direito a algo?", "tool": False},
    {"q": "Como funciona o cancelamento de um pedido que ainda está em preparo?", "tool": False},
    {"q": f"Pode me dizer o status do pedido {status_order_id}?", "tool": True},
    {
        "q": f"Quero abrir um reembolso do pedido {refund_order['id']}: o item "
        f"'{refund_item}' não veio. Pode abrir agora, por favor.",
        "tool": True,
    },
]


def run_smoke(llm, check_alive=None) -> list[dict]:
    """Run the 5 questions through the real agent loop."""
    rows = []
    for item in QUESTIONS:
        if check_alive:
            check_alive()
        ctx = ToolContext(conn=conn, customer_id=customer_id, now=REFERENCE_NOW)
        cfg = AgentConfig(retrieval="bm25", defense="full")
        began = time.perf_counter()
        res = run_agent(
            [{"role": "user", "content": item["q"]}], ctx, cfg, retrieve, llm, CANARY, NONCE
        )
        latency = time.perf_counter() - began
        llm_spans = [s for s in res.trace["spans"] if s["name"] == "llm_call"]
        row = {
            "question": item["q"],
            "expects_tool": item["tool"],
            "answer": res.answer,
            "sources": res.sources,
            "tool_outcomes": [(o.name, o.args, o.result) for o in res.tool_outcomes],
            "latency_s": latency,
            "prompt_tokens": sum(s["prompt_tokens"] for s in llm_spans),
            "completion_tokens": sum(s["completion_tokens"] for s in llm_spans),
        }
        rows.append(row)
        print("=" * 80)
        for key, value in row.items():
            print(f"{key}: {value}")
    return rows

In [ ]:
llm = OpenAIChatClient(base_url=f"http://localhost:{PORT}/v1", model=MODEL)
try:
    rows = run_smoke(llm, check_alive=ensure_server_alive)
except Exception:
    print(log_tail())
    raise

In [ ]:
import statistics

import vllm


def summarize(rows: list[dict], vllm_version: str, model: str) -> None:
    latencies = [r["latency_s"] for r in rows]
    tool_rows = [r for r in rows if r["expects_tool"]]
    parsed = [bool(r["tool_outcomes"]) for r in tool_rows]
    cited = [bool(r["sources"]) for r in rows if not r["expects_tool"]]
    print(f"vllm version: {vllm_version}")
    print(f"model id: {model}")
    print(f"latency mean/max (s): {statistics.fmean(latencies):.1f} / {max(latencies):.1f}")
    print(
        f"tokens prompt/completion: {sum(r['prompt_tokens'] for r in rows)} / "
        f"{sum(r['completion_tokens'] for r in rows)}"
    )
    print(f"tool_calls parsed on tool questions: {sum(parsed)}/{len(parsed)}")
    print(f"policy answers citing a known [passage_id]: {sum(cited)}/{len(cited)}")


summarize(rows, vllm.__version__, MODEL)

In [ ]:
server.terminate()
try:
    server.wait(timeout=60)
except subprocess.TimeoutExpired:
    server.kill()
log_file.close()
print("vLLM stopped")